## What Drives Website Visits? Three ML Models

In [1]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

sys.path.insert(0, "../src")
from uplift import make_features

df = pd.read_csv("../data/hillstrom.csv")
X = make_features(df)
X["mens_email"] = (df["segment"] == "Mens E-Mail").astype(int)
X["womens_email"] = (df["segment"] == "Womens E-Mail").astype(int)
y = df["visit"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(X_tr.shape, X_te.shape)
X.head()

(44800, 11) (19200, 11)


,recency,history,mens,womens,newbie,zip_code_Surburban,zip_code_Urban,channel_Phone,channel_Web,mens_email,womens_email
0,10,142.44,1,0,0,1,0,1,0,0,1
1,6,329.08,1,1,1,0,0,0,1,0,0
2,7,180.65,0,1,1,1,0,0,1,0,1
3,9,675.83,1,0,1,0,0,0,1,1,0
4,2,45.34,1,0,0,0,1,0,1,0,1


In [2]:
models = {
    "Logistic regression (baseline)": LogisticRegression(max_iter=2000),
    "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42),
}
auc = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    auc[name] = roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
pd.Series(auc, name="Test ROC AUC").round(3)

Logistic regression (baseline)    0.648
Decision tree (depth 4)           0.618
XGBoost                           0.648
Name: Test ROC AUC, dtype: float64

## Model comparison
Test ROC AUC: logistic regression 0.648, decision tree 0.618, XGBoost 0.648. XGBoost matched but did not beat the simple baseline, which tells us the extra complexity is not worth it with these few features: the relationships are mostly simple and additive. The shallow decision tree did worst because a 4-level flowchart is too coarse. With equal accuracy, logistic regression is the better choice because each coefficient can be explained to a business audience, which is what the next section does.